# Atomic requirements R5 custody replication
Checks the R5 successor intake, including the 46 semantic acceptance atoms. Models and app integrations remain NOT_EXECUTED.


In [ ]:
from pathlib import Path
import json, hashlib, subprocess, sys
ROOT=Path.cwd()
if not (ROOT/'sources/challenge.pdf').exists(): ROOT=ROOT.parent
assert (ROOT/'sources/challenge.pdf').exists()
expected='f9c730d4c6464cdfeedb540e0d4b6080b7bfcde9fab496a1eac0dbc15b0aa1cc'
assert hashlib.sha256((ROOT/'sources/challenge.pdf').read_bytes()).hexdigest()==expected
atoms=[json.loads(x) for x in (ROOT/'data/rules/atoms.jsonl').read_text().splitlines()]
content=json.loads((ROOT/'data/rules/deduplicated_content.json').read_text())
assert all(content[a['content_sha256']]==a['text'] for a in atoms)
print(json.dumps({'source_identity':True,'atoms':len(atoms),'occurrence_reconstruction_exact':True}))

In [ ]:
rows=[json.loads(x) for x in (ROOT/'data/rules/atomic_acceptance_r5.jsonl').read_text().splitlines()]
source=(ROOT/'sources/user_rules_excerpt_20261020.txt').read_text()
for row in rows:
    ctx=row['occurrence']; assert source[ctx['start_codepoint']:ctx['end_codepoint']]==row['source_quote']
    payload={k:v for k,v in row.items() if k!='fco_candidate_sha256'}
    encoded=json.dumps(payload,sort_keys=True,ensure_ascii=True,separators=(',',':')).encode('ascii')
    assert hashlib.sha256(encoded).hexdigest()==row['fco_candidate_sha256']
    assert row['verified'] is False and row['observed_outcome']=='NOT_TESTED'
assert len(rows)==46 and sum(r['required_for_entry'] for r in rows)==30
print(json.dumps({'atoms':46,'mandatory':30,'source_bindings_valid':True,'acceptance_gate':False}))


## Recompute candidate custody

SHA-256, domain-separated leaves/internal nodes, UTF-8 path ordering and odd-node promotion are declared in the receipt. Expected adversarial acceptance results are FALSE.

In [ ]:
r=subprocess.run([sys.executable,str(ROOT/'scripts/recompute_requirements_r5.py')],cwd=ROOT,text=True,capture_output=True,check=True)
print(r.stdout)
receipt=json.loads((ROOT/'receipts/INTAKE_MERKLE_R5.json').read_text())
assert receipt['checks']['all_inclusion_proofs_valid'] is True
assert receipt['checks']['tampered_leaf_accepted'] is False

In [ ]:
r=subprocess.run(['node',str(ROOT/'scripts/verify_requirements_r5.mjs')],cwd=ROOT,text=True,capture_output=True,check=True)
print(r.stdout)
result=json.loads(r.stdout)
assert result['root']==receipt['root'] and result['result']=='PASS'

## Limits

PASS applies to the declared intake snapshot only. Model/VM/BAND/Cognee/Snyk/Flexprice experiments remain NOT_EXECUTED. Project MMR NOT_COMMITTED; SIGNED=NO; Anticube and DeltaG* NOT_COMPUTED. Four synthetic fixtures are development data, not a validated adversarial benchmark.